<a href="https://colab.research.google.com/github/Wapikkk/text-summarize-nlp/blob/extractive/notebooks/02_extractive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 - Extractive Summarization

**Tujuan:** membuat ringkasan dengan memilih kalimat penting dari teks asli (tanpa menulis ulang) memakai TF-IDF, TextRank, dan LSA, dengan baseline lead-k sebagai pembanding.

**Output:** file prediksi JSONL per metode di `results/` (kolom `id`, `reference`, `prediction`), beserta waktu proses.



## **Install Package**
Memasang library untuk stemming bahasa Indonesia (Sastrawi), ROUGE, graf (networkx), dan TF-IDF/LSA (scikit-learn).

In [1]:
!pip -q install Sastrawi rouge-score networkx scikit-learn datasets

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## **Import Package**
Mengimpor library yang dipakai, menghubungkan Google Drive, dan mendefinisikan `read_jsonl` untuk membaca file split.

In [2]:
import re, json, os, time
import numpy as np
import networkx as nx

from functools import lru_cache
from google.colab import drive

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

from rouge_score import rouge_scorer

drive.mount('/content/drive')

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

Mounted at /content/drive


## **Load Dataset**
Membaca `train.jsonl`, `val.jsonl`, dan `test.jsonl` hasil 01_data dari folder Drive (`dokumen_nlp`): 37.970 / 4.725 / 4.778 bagian. Train dipakai untuk menghitung IDF, val untuk menyetel parameter, dan test hanya untuk hasil akhir.

In [3]:
USE_OFFICIAL = False
DATA_DIR = '/content/drive/MyDrive/dokumen_nlp' # directory hasil preprocessing 01_data

train = read_jsonl(f'{DATA_DIR}/train.jsonl')
val   = read_jsonl(f'{DATA_DIR}/val.jsonl')
test  = read_jsonl(f'{DATA_DIR}/test.jsonl')

print('Informasi hasil preprocessing 01_data:')
print('  train:', len(train))
print('  val  :', len(val))
print('  test :', len(test))

Informasi hasil preprocessing 01_data:
  train: 37970
  val  : 4725
  test : 4778


## **Pemisahan Kalimat**
Dokumen dipecah menjadi kalimat dengan regex: pemisah adalah tanda `. ! ?` yang diikuti huruf kapital, angka, atau tanda kutip. Kalimat kurang dari 3 kata dibuang. Di data val, jumlah kalimat per dokumen adalah median 21 (p10 = 8, p90 = 42), dan panjang ringkasan referensi median 27 kata (p10 = 13, p90 = 54). Ini menjadi dasar memilih jumlah kalimat `k` yang kecil.

In [4]:
def split_sentences(text):
    text = re.sub(r'\s+', ' ', text).strip()
    sents = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"“(])', text)
    sents = [s for s in sents if len(s.split()) >= 3]
    return sents or [text]

n_sent  = [len(split_sentences(r['document'])) for r in val]
ref_len = [len(r['summary'].split()) for r in val]
print('kalimat/dokumen (p10, p50, p90):', np.percentile(n_sent,  [10, 50, 90]))
print('kata/ringkasan  (p10, p50, p90):', np.percentile(ref_len, [10, 50, 90]))

kalimat/dokumen (p10, p50, p90): [ 8. 21. 42.]
kata/ringkasan  (p10, p50, p90): [13. 27. 54.]


## **Preprocessing untuk Scorring**
Setiap kalimat diubah menjadi huruf kecil, hanya menyisakan token huruf dan angka, stopword dibuang (daftar Sastrawi), lalu di-stem dengan Sastrawi (hasil stem disimpan dalam cache agar lebih cepat). Preprocessing ini hanya untuk menghitung skor.

In [16]:
STOP = set(StopWordRemoverFactory().get_stop_words())

@lru_cache(maxsize=None)
def prep(sentence):
    toks = re.findall(r'[a-z0-9]+', sentence.lower())
    return ' '.join(t for t in toks if t not in STOP)

## **Metode Extractive dan Baseline**
Setiap metode memberi skor pada tiap kalimat, lalu mengambil `k` kalimat teratas dalam urutan aslinya. Jika jumlah kalimat tidak lebih dari `k`, semua kalimat dikembalikan.
- **TF-IDF:** skor kalimat = rata-rata bobot TF-IDF kata di kalimat itu (IDF dihitung dari 3.000 dokumen train pertama).
- **TextRank:** kalimat menjadi simpul graf dengan bobot kemiripan kosinus, lalu diberi skor dengan PageRank.
- **LSA:** SVD pada matriks kalimat-kata TF-IDF (3 komponen); skor kalimat = panjang vektornya di ruang laten.
- **Lead-k (baseline):** `k` kalimat pertama.

In [17]:
def pick(scores, sents, k):
    idx = np.argsort(scores)[::-1][:k]
    return ' '.join(sents[i] for i in sorted(idx))

# TF-IDF (IDF dihitung dari dokumen train)
vec = TfidfVectorizer(norm=None).fit([prep(r['document']) for r in train[:2000]])

def tfidf_summ(doc, k):
    sents = split_sentences(doc)
    if len(sents) <= k: return ' '.join(sents)
    X = vec.transform([prep(s) for s in sents])
    scores = np.asarray(X.sum(axis=1)).ravel() / np.maximum(X.getnnz(axis=1), 1)
    return pick(scores, sents, k)

# TextRank
def textrank_summ(doc, k):
    sents = split_sentences(doc)
    if len(sents) <= k: return ' '.join(sents)
    try:
        X = TfidfVectorizer().fit_transform([prep(s) for s in sents])
    except ValueError:
        return ' '.join(sents[:k])
    sim = cosine_similarity(X)
    np.fill_diagonal(sim, 0)
    try:
        pr = nx.pagerank(nx.from_numpy_array(sim), max_iter=200)
    except nx.PowerIterationFailedConvergence:
        return ' '.join(sents[:k])
    return pick([pr[i] for i in range(len(sents))], sents, k)

# LSA
def lsa_summ(doc, k, n_comp=3):
    sents = split_sentences(doc)
    if len(sents) <= k: return ' '.join(sents)
    try:
        X = TfidfVectorizer().fit_transform([prep(s) for s in sents])
    except ValueError:
        return ' '.join(sents[:k])
    nc = min(n_comp, X.shape[0] - 1, X.shape[1] - 1)
    if nc < 1: return ' '.join(sents[:k])
    Z = TruncatedSVD(n_components=nc, random_state=42).fit_transform(X)
    return pick(np.sqrt((Z ** 2).sum(axis=1)), sents, k)

# Lead-k (baseline)
def lead_summ(doc, k):
    return ' '.join(split_sentences(doc)[:k])

methods = {
    'tfidf': tfidf_summ,
    'textrank': textrank_summ,
    'lsa': lsa_summ,
    'lead': lead_summ,
}

## **Contoh Hasil Ringkasan**
Satu dokumen val diringkas dengan tiap metode (`k = 2`) dan dibandingkan dengan ringkasan referensi, untuk pemeriksaan kualitas secara kasat mata.

In [11]:
doc = val[0]['document']
print('DOKUMEN  :', doc[:400], '\n')
print('REFERENSI:', val[0]['summary'], '\n')
for name, fn in methods.items():
    print(name.upper(), ':', fn(doc, 2), '\n')

DOKUMEN  : Di musim tertentu, buah dan sayuran dijual dengan harga yang lebih murah. Sering kali, swalayan memberi diskon khusus untuk musim tertentu karena para pembeli mencari buah dan sayuran yang spesifik pada waktu tersebut. Sebagai bonus, saat sedang musimnya, buah dan sayuran yang dijual terasa lebih lezat. Di musim gugur, carilah produk seperti labu, squash musim dingin, kol, dan apel. Di musim dingi 

REFERENSI: Belilah bahan makanan sesuai musimnya. Periksalah harga makanan beku dan kalengan. Periksa diskon mingguan. Selalu bandingkan harga jual. Belilah dalam jumlah besar. Gunakanlah kupon. Di Amerika Serikat, Anda dapat mendaftarkan diri untuk program SNAP atau WIC. Carilah pasar atau peternakan terdekat. 

TFIDF : Di musim dingin, carilah squash musim dingin, kol, serta umbi-umbian lainnya. Manfaatkanlah kupon jika Anda memilikinya. 

TEXTRANK : Di musim tertentu, buah dan sayuran dijual dengan harga yang lebih murah. Selain itu, karena ada banyak produk dijual murah di mu

## **Fungsi Evaluasi ROUGE**
Menghitung ROUGE-1, ROUGE-2, dan ROUGE-L (F1) dengan `rouge_score`, lalu memberikan rata-rata di seluruh contoh. Perhitungan akhir akan dilakukan di 04_evalutaion dengan skrip bersama.

In [12]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'])

def mean_rouge(preds, refs):
    out = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for p, r in zip(preds, refs):
        s = scorer.score(r, p)
        for m in out:
            out[m].append(s[m].fmeasure)
    return {m: round(float(np.mean(v)), 4) for m, v in out.items()}

## **Konfigurasi k di Data Val**
Setiap metode dicoba dengan `k` = 1 sampai 4 pada 1.000 contoh val. Dicatat ROUGE dan panjang rata-rata ringkasan. `k` terbaik per metode dipilih dari ROUGE-1 F1. Karena ROUGE naik seiring ringkasan memanjang, versi `k = 2` (panjang mendekati referensi, sekitar 30 kata) juga disimpan sebagai pembanding yang adil bagi model abstractive.

In [13]:
val_s = val[:1000]
refs = [r['summary'] for r in val_s]
print('rata-rata panjang referensi (kata):', round(np.mean([len(x.split()) for x in refs]), 1))

results_val = {}
for name, fn in methods.items():
    for k in [1, 2, 3, 4]:
        preds = [fn(r['document'], k) for r in val_s]
        results_val[(name, k)] = {**mean_rouge(preds, refs),
                                  'len': round(np.mean([len(p.split()) for p in preds]), 1)}
        print(name, '| k =', k, '|', results_val[(name, k)])

BEST_K = {name: max([1, 2, 3, 4], key=lambda k: results_val[(name, k)]['rouge1']) for name in methods}
print('k terbaik (ROUGE-1 F1 di val):', BEST_K)

rata-rata panjang referensi (kata): 32.7
tfidf | k = 1 | {'rouge1': 0.1307, 'rouge2': 0.0288, 'rougeL': 0.1072, 'len': np.float64(16.2)}
tfidf | k = 2 | {'rouge1': 0.1809, 'rouge2': 0.0408, 'rougeL': 0.1369, 'len': np.float64(32.6)}
tfidf | k = 3 | {'rouge1': 0.2011, 'rouge2': 0.0478, 'rougeL': 0.1466, 'len': np.float64(48.8)}
tfidf | k = 4 | {'rouge1': 0.2046, 'rouge2': 0.0503, 'rougeL': 0.1462, 'len': np.float64(64.3)}
textrank | k = 1 | {'rouge1': 0.1969, 'rouge2': 0.0511, 'rougeL': 0.1514, 'len': np.float64(18.0)}
textrank | k = 2 | {'rouge1': 0.2331, 'rouge2': 0.0628, 'rougeL': 0.1699, 'len': np.float64(35.0)}
textrank | k = 3 | {'rouge1': 0.2355, 'rouge2': 0.0646, 'rougeL': 0.1687, 'len': np.float64(51.4)}
textrank | k = 4 | {'rouge1': 0.2299, 'rouge2': 0.0648, 'rougeL': 0.1632, 'len': np.float64(67.3)}
lsa | k = 1 | {'rouge1': 0.1481, 'rouge2': 0.0349, 'rougeL': 0.1176, 'len': np.float64(14.7)}
lsa | k = 2 | {'rouge1': 0.1888, 'rouge2': 0.0441, 'rougeL': 0.1434, 'len': np.float6

## **Konfigurasi LSA**
Jumlah komponen SVD dicoba 2, 3, dan 5 dengan `k = 2`. Jika nilai terbaik berbeda dari bawaan (3), ubah nilai bawaan `n_comp` di fungsi `lsa_summ`.

In [14]:
for nc in [2, 3, 5]:
    preds = [lsa_summ(r['document'], 2, n_comp=nc) for r in val_s]
    print('LSA n_comp =', nc, mean_rouge(preds, refs))

LSA n_comp = 2 {'rouge1': 0.1941, 'rouge2': 0.0485, 'rougeL': 0.1478}
LSA n_comp = 3 {'rouge1': 0.1888, 'rouge2': 0.0441, 'rougeL': 0.1434}
LSA n_comp = 5 {'rouge1': 0.1791, 'rouge2': 0.0406, 'rougeL': 0.1371}


## **Menyimpan Hasil Prediksi Test**
Tiap metode dijalankan pada seluruh data test dalam dua versi: `k` terbaik dari val dan `k = 2`. Hasilnya disimpan sebagai JSONL (`id`, `reference`, `prediction`) di `dokumen_nlp/results/`, beserta waktu inferensi tiap metode di `extractive_run_info.json`.

In [15]:
RESULT_DIR = f'{DATA_DIR}/results'
os.makedirs(RESULT_DIR, exist_ok=True)
run_info = {}

for name, fn in methods.items():
    for suffix, k in [('', BEST_K[name]), ('_k2', 2)]:
        t0 = time.time()
        preds = [fn(r['document'], k) for r in test]
        dt = time.time() - t0
        path = f'{RESULT_DIR}/extractive_{name}{suffix}.jsonl'
        with open(path, 'w', encoding='utf-8') as f:
            for r, p in zip(test, preds):
                f.write(json.dumps({'id': r['id'], 'reference': r['summary'], 'prediction': p},
                                   ensure_ascii=False) + '\n')
        run_info[f'{name}{suffix}'] = {'k': k, 'infer_time_s': round(dt, 1), 'n_test': len(test)}
        print(name, suffix, 'k =', k, '|', round(dt, 1), 'detik')

with open(f'{RESULT_DIR}/extractive_run_info.json', 'w') as f:
    json.dump(run_info, f, indent=2)

tfidf  k = 4 | 5.4 detik
tfidf _k2 k = 2 | 4.3 detik
textrank  k = 3 | 27.5 detik
textrank _k2 k = 2 | 27.6 detik
lsa  k = 4 | 21.1 detik
lsa _k2 k = 2 | 23.2 detik
lead  k = 4 | 1.1 detik
lead _k2 k = 2 | 1.1 detik
